# 🧪 Pipeline Tester — No LiveKit Required

Notebook này chạy toàn bộ pipeline **STT → LLM → TTS** trên Colab GPU T4,  
expose ra ngoài qua **ngrok tunnel**, và bạn dùng file `test_ui.html` ở local để test.

**Architecture:**
```
Browser (test_ui.html) ──HTTP/WS──► ngrok ──► Colab (test_server.py)
                                                    ├── Whisper (STT)
                                                    ├── Ollama  (LLM)
                                                    └── Edge-TTS (TTS)
```

## Cell 1 — Install hệ thống (chạy 1 lần)

In [ ]:
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ ffmpeg installed')

%pip install -q \
    fastapi uvicorn[standard] \
    openai-whisper \
    edge-tts \
    pydub \
    httpx \
    pyngrok \
    numpy

print('✅ Python deps installed')

## Cell 2 — Cài & khởi động Ollama + model

In [ ]:
import subprocess, time, os

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
print('✅ Ollama installed')

# Start Ollama server in background
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env={**os.environ, 'OLLAMA_HOST': '0.0.0.0'}
)
time.sleep(4)
print('✅ Ollama server started (PID', proc.pid, ')')

# Pull translation model — gemma2:2b is fast on T4
# Change to 'gemma2:9b' or 'llama3.1:8b' for better quality
MODEL = 'gemma2:2b'
print(f'Pulling {MODEL}... (may take a few minutes)')
!ollama pull {MODEL}
print(f'✅ {MODEL} ready')

## Cell 3 — Viết file server

In [ ]:
server_code = '''
import asyncio, io, json, logging, os, tempfile, time, base64
import httpx, whisper, edge_tts
from fastapi import FastAPI, File, Form, UploadFile, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydub import AudioSegment

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("test-server")

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

OLLAMA_URL   = os.getenv("OLLAMA_URL",   "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma2:2b")
SYSTEM_PROMPT = (
    "You are a professional interpreter. "
    "Translate the following Vietnamese speech into natural English. "
    "Output ONLY the English translation — no notes, no explanations."
)

_whisper = None
def get_whisper():
    global _whisper
    if _whisper is None:
        logger.info("Loading Whisper base model...")
        _whisper = whisper.load_model("base")
    return _whisper

async def run_stt(audio_bytes, fmt="webm"):
    with tempfile.NamedTemporaryFile(suffix=f".{fmt}", delete=False) as tmp:
        tmp.write(audio_bytes); tmp_path = tmp.name
    try:
        seg = AudioSegment.from_file(tmp_path)
        seg = seg.set_channels(1).set_frame_rate(16000)
        wav = tmp_path + ".wav"
        seg.export(wav, format="wav")
        result = get_whisper().transcribe(wav, language="vi", task="transcribe")
        return result["text"].strip()
    finally:
        for p in [tmp_path, tmp_path + ".wav"]:
            try: os.remove(p)
            except: pass

async def run_llm(text):
    async with httpx.AsyncClient(timeout=60) as c:
        r = await c.post(f"{OLLAMA_URL}/api/chat", json={
            "model": OLLAMA_MODEL, "stream": False,
            "messages": [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":text}]
        })
        r.raise_for_status()
        return r.json()["message"]["content"].strip()

async def run_tts(text, rate="+15%"):
    com = edge_tts.Communicate(text, voice="en-US-JennyNeural", rate=rate)
    buf = io.BytesIO()
    async for chunk in com.stream():
        if chunk["type"] == "audio": buf.write(chunk["data"])
    return buf.getvalue()

@app.get("/health")
def health(): return {"status": "ok", "model": OLLAMA_MODEL}

@app.post("/api/transcribe")
async def transcribe(file: UploadFile = File(...)):
    t0 = time.perf_counter()
    audio = await file.read()
    fmt = file.filename.rsplit(".", 1)[-1] if "." in file.filename else "webm"
    text = await run_stt(audio, fmt)
    return {"text": text, "latency_ms": round((time.perf_counter()-t0)*1000)}

@app.post("/api/translate")
async def translate(text: str = Form(...)):
    t0 = time.perf_counter()
    result = await run_llm(text)
    return {"translation": result, "latency_ms": round((time.perf_counter()-t0)*1000)}

@app.post("/api/speak")
async def speak(text: str = Form(...)):
    mp3 = await run_tts(text)
    return StreamingResponse(io.BytesIO(mp3), media_type="audio/mpeg")

@app.post("/api/pipeline")
async def full_pipeline(file: UploadFile = File(...)):
    timings = {}
    audio = await file.read()
    fmt = file.filename.rsplit(".", 1)[-1] if "." in file.filename else "webm"
    t0 = time.perf_counter(); vn_text = await run_stt(audio, fmt); timings["stt_ms"] = round((time.perf_counter()-t0)*1000)
    t0 = time.perf_counter(); en_text = await run_llm(vn_text);    timings["llm_ms"] = round((time.perf_counter()-t0)*1000)
    t0 = time.perf_counter(); mp3     = await run_tts(en_text);    timings["tts_ms"] = round((time.perf_counter()-t0)*1000)
    return {"stt": vn_text, "translation": en_text, "audio_b64": base64.b64encode(mp3).decode(), "timings": timings}

@app.websocket("/ws")
async def websocket_pipeline(ws: WebSocket):
    await ws.accept()
    try:
        while True:
            msg = json.loads(await ws.receive_text())
            if msg.get("type") != "audio": continue
            audio_bytes = base64.b64decode(msg["data"])
            fmt = msg.get("format", "webm")
            await ws.send_text(json.dumps({"type":"status","text":"Transcribing..."}))
            vn = await run_stt(audio_bytes, fmt)
            await ws.send_text(json.dumps({"type":"stt","text":vn}))
            await ws.send_text(json.dumps({"type":"status","text":"Translating..."}))
            en = await run_llm(vn)
            await ws.send_text(json.dumps({"type":"translation","text":en}))
            await ws.send_text(json.dumps({"type":"status","text":"Synthesizing speech..."}))
            mp3 = await run_tts(en)
            await ws.send_text(json.dumps({"type":"audio","data":base64.b64encode(mp3).decode()}))
            await ws.send_text(json.dumps({"type":"done"}))
    except WebSocketDisconnect: pass
    except Exception as e:
        await ws.send_text(json.dumps({"type":"error","text":str(e)}))
'''

with open('/content/test_server.py', 'w') as f:
    f.write(server_code)
print('✅ test_server.py written')

## Cell 4 — Cấu hình ngrok token

Lấy token miễn phí tại https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # 👈 Thay token vào đây

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
print('✅ ngrok token set')

## Cell 5 — Khởi động server + lấy public URL

In [ ]:
import subprocess, threading, time, requests
from pyngrok import ngrok

# Kill any existing server
!pkill -f test_server.py 2>/dev/null; sleep 1

# Start FastAPI in background thread
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'test_server:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Wait for server to be ready
print('Waiting for server...')
for _ in range(20):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.ok:
            print('✅ Server is up:', r.json())
            break
    except:
        time.sleep(1)
else:
    print('❌ Server failed to start! Check logs:')
    print(server_proc.stdout.read(2048).decode())

# Open ngrok tunnel
tunnel = ngrok.connect(8000, bind_tls=True)
PUBLIC_URL = tunnel.public_url

print('\n' + '='*60)
print('🌐 PUBLIC URL:', PUBLIC_URL)
print('='*60)
print('\n👆 Copy URL này vào ô "Server URL" trong test_ui.html')
print('\nEndpoints:')
print(f'  POST {PUBLIC_URL}/api/transcribe  — STT only')
print(f'  POST {PUBLIC_URL}/api/translate   — LLM only')
print(f'  POST {PUBLIC_URL}/api/speak       — TTS only')
print(f'  POST {PUBLIC_URL}/api/pipeline    — Full pipeline')
print(f'  WS   {PUBLIC_URL.replace("https","wss")}/ws — Real-time WebSocket')

## Cell 6 — (Optional) Test nhanh bằng Python

In [ ]:
import requests

# Test LLM translation trực tiếp
text = "Xin chào, tôi muốn học lập trình Python để làm AI."
print('Input:', text)

r = requests.post('http://localhost:8000/api/translate', data={'text': text})
data = r.json()
print('Translation:', data['translation'])
print('Latency:', data['latency_ms'], 'ms')

In [ ]:
# Test TTS — download file MP3
import requests

en_text = "Hello, I want to learn Python programming for AI."
r = requests.post('http://localhost:8000/api/speak', data={'text': en_text})

with open('/content/output.mp3', 'wb') as f:
    f.write(r.content)

# Play in Colab
from IPython.display import Audio
Audio('/content/output.mp3')

In [ ]:
# Test full pipeline với file audio
# Tạo audio test từ Google TTS
import requests, subprocess

# Record 5 giây từ mic nếu có, hoặc dùng file có sẵn
# Cách đơn giản: dùng edge-tts tạo audio tiếng Việt để test
import asyncio, edge_tts

async def make_test_audio():
    com = edge_tts.Communicate(
        "Xin chào, tôi tên là Nam, tôi đến từ Thành phố Hồ Chí Minh.",
        voice="vi-VN-NamMinhNeural"
    )
    await com.save('/content/test_input.mp3')

await make_test_audio()
print('✅ Test audio created: test_input.mp3')

# Run through full pipeline
with open('/content/test_input.mp3', 'rb') as f:
    r = requests.post('http://localhost:8000/api/pipeline',
                      files={'file': ('test.mp3', f, 'audio/mpeg')})

data = r.json()
print('\n📝 STT (Vietnamese):', data['stt'])
print('🌐 Translation (English):', data['translation'])
print('⏱  Timings:', data['timings'])

# Play TTS output
import base64
from IPython.display import Audio
mp3_bytes = base64.b64decode(data['audio_b64'])
with open('/content/output_translation.mp3', 'wb') as f:
    f.write(mp3_bytes)
Audio('/content/output_translation.mp3')

## Cell 7 — Dừng server

In [ ]:
from pyngrok import ngrok
ngrok.kill()
server_proc.terminate()
print('✅ Server stopped')